# Model Training - Appartement Vente Marrakech
Ce notebook utilise les fonctions de `model_training_helpers.py` pour entraîner et évaluer un modèle **Stacking** avec transformation logarithmique du prix.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor, StackingRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.compose import TransformedTargetRegressor

# Ajouter le chemin vers la racine pour importer les helpers
sys.path.append('../')
from model_training_helpers import split_data, model_pipeline, metric_model, get_features, print_metrics, save_model, tune_model, grid_search_model, preprocess_data

print("Modules chargés avec succès.")

Modules chargés avec succès.


## 1. Chargement des données et fusion NLP

In [2]:
csv_path = '../data/cleaned_data/vente/appartement_vente_cleaned.csv'
raw_csv_path = '../data/marrakech_immo_vente/appartement_vente.csv'

if os.path.exists(csv_path) and os.path.exists(raw_csv_path):
    df = pd.read_csv(csv_path)
    raw_df = pd.read_csv(raw_csv_path)
    
    # Merging the description from the raw data
    merge_cols = ['prix_num', 'surface_num', 'chambres_num', 'salles_bain_num', 'quartier']
    raw_subset = raw_df[merge_cols + ['description']].drop_duplicates(subset=merge_cols)
    df = pd.merge(df, raw_subset, on=merge_cols, how='left')
    
    print(f"Données chargées : {df.shape}, avec descriptions ajoutées.")
else:
    print(f"ERREUR : Fichiers non trouvés.")

Données chargées : (5448, 26), avec descriptions ajoutées.


## 2. Préparation et Split

In [3]:
# Appliquer le feature engineering (inclus l'extraction de texte et Isolation Forest)
df = preprocess_data(df, property_type='appartement')

# Utilisation de la fonction split_data
X_train, X_test, y_train, y_test = split_data(df, target='prix_num')

# Identification automatique des colonnes
num_features, cat_features = get_features(X_train)

print(f"\nVariables numériques ({len(num_features)}) : {num_features}")
print(f"Variables catégorielles ({len(cat_features)}) : {cat_features}")


Variables numériques (36) : ['piscine', 'parking', 'ascenseur', 'terrasse', 'jardin', 'climatisation', 'securite', 'vue', 'meuble', 'neuf', 'cave', 'hammam', 'surface_num', 'chambres_num', 'salles_bain_num', 'nb_pieces', 'prix_m2_median_quartier', 'log_surface_num', 'etage_num', 'surface_par_chambre', 'score_commodites', 'surface_score_interaction', 'is_luxury_location', 'prix_estime_quartier', 'kw_luxe', 'kw_standing', 'kw_neuf', 'kw_rénové', 'kw_moderne', 'kw_calme', 'kw_vue_atlas', 'kw_piscine', 'kw_sécurisée', 'kw_ascenseur', 'kw_parking', 'kw_ensoleillé']
Variables catégorielles (3) : ['agence', 'etage', 'quartier']


## 3. Entraînement du modèle Stacking + Log Target

In [4]:
print("Entraînement d'un modèle Stacking + Log-Target pour atteindre >80%...")

xgb_model = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.01, max_depth=9, subsample=0.8, colsample_bytree=0.8, random_state=42)
rf_model = RandomForestRegressor(n_estimators=500, max_depth=15, random_state=42)
hgb_model = HistGradientBoostingRegressor(max_iter=1000, max_depth=10, random_state=42)

estimators = [
    ('xgb', xgb_model),
    ('rf', rf_model),
    ('hgb', hgb_model)
]

stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(),
    cv=3
)

# Pipeline de base
base_pipeline = model_pipeline(num_features, cat_features, stacking_model)

# Ajout de la transformation logarithmique de la cible
pipeline = TransformedTargetRegressor(
    regressor=base_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

print("Lancement de l'entraînement final (cela peut prendre 10-15 minutes)... ")
pipeline.fit(X_train, y_train)
print("Modèle final entraîné avec succès.")

Entraînement d'un modèle Stacking + Log-Target pour atteindre >80%...
Lancement de l'entraînement final (cela peut prendre 10-15 minutes)... 


Modèle final entraîné avec succès.


## 4. Évaluation

In [5]:
y_pred = pipeline.predict(X_test)

# Calcul des métriques
metrics = metric_model(y_test, y_pred, model_name="Final Stacking Log")

# Affichage
print_metrics(metrics, model_name="Final Stacking Log")


--- Évaluation des performances : Final Stacking Log ---
MAE (Erreur Absolue Moyenne) : 163,158.85 DH
RMSE (Racine de l'Erreur Quadratique Moyenne) : 229,294.46 DH
Score R² : 0.8031


## 5. Sauvegarde

In [6]:
save_model(pipeline, 'appartement_vente_final_model.joblib')

Modèle sauvegardé dans : models/appartement_vente_final_model.joblib
